### NLP Airbnb Data

The purpose of this code is to conduct some initial exploratory analysis and NLP experiments. 

This file should be run on the output from the 'airbnb-processing' code.

In [ ]:
# imports
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk import bigrams
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from nltk.tokenize import word_tokenize
nltk.download('punkt')

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

In [ ]:
# To run this code on a different set of data, change the file name below
file_name = "output.csv"
df = pd.read_csv(file_name)

### EDA: Exploratory Analysis of the AirBNB dataset

In [ ]:
df[['description']].head()

In [ ]:
df['char_length'] = df['description'].str.len()
df['word_count'] = df['description'].str.split().str.len()

In [ ]:
# Examing the distribution of word count across the listings 

plt.hist(df['word_count'], bins=50)
plt.xlabel("Word Count")
plt.ylabel("Number of Listings")
plt.title("Distribution of Description Lengths")
plt.show()

In [ ]:
# print number of null descriptions
df['description'].isna().sum()
# drop all null descriptions 
df = df.dropna(subset=['description'])

Question for team: What should we do if we don't have a description? Shoudl we look at any other text data associated with the reocrd (host description, reviews, etc.)? 

In [ ]:
# text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)              # remove numbers
    text = re.sub(r'[^\w\s]', '', text)          # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()     # remove extra spaces
    return text

In [ ]:
df['clean_description'] = df['description'].apply(clean_text)

In [ ]:
# check to make sure we did not remove valuable information 
df[['description', 'clean_description']].sample(5)

In [ ]:
# tokenization  
df['tokens'] = df['clean_description'].apply(word_tokenize)

In [ ]:
df['token_count'] = df['tokens'].apply(len)
df['token_count_nostop'] = df['tokens_nostop'].apply(len)

In [ ]:
# examine token coutns before and after stopword removal 

## Stopwords = the, and, is, or, to.... etc.
plt.hist(df['token_count'], bins=50, alpha=0.6, label="Raw Tokens")
plt.hist(df['token_count_nostop'], bins=50, alpha=0.6, label="No Stopwords")
plt.legend()
plt.title("Token Counts Before and After Stopword Removal")
plt.show()

In [ ]:
# Stop word removal
stop_words = set(stopwords.words('english'))

df['tokens_nostop'] = df['tokens'].apply(
    lambda x: [word for word in x if word not in stop_words]
)

In [ ]:
# Examine the most common words
all_tokens = [word for tokens in df['tokens'] for word in tokens]
all_tokens_nostop = [word for tokens in df['tokens_nostop'] for word in tokens]

Counter(all_tokens).most_common(20)
Counter(all_tokens_nostop).most_common(20)

In [ ]:
# convert words to base word (ex. walking -> walk)
lemmatizer = WordNetLemmatizer()

df['tokens_lemma'] = df['tokens_nostop'].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)

Note: I am having an issue with lematizatiom -- from my readings we should see more change and a bigger dip. Further investigation is needed.

In [ ]:
## EXamples of what we are cleaning up:: blocks -> block, stores -> store
sample_idx = df.sample(1).index[0]

print("Before:", df.loc[sample_idx, 'tokens_nostop'][:20])
print("After :", df.loc[sample_idx, 'tokens_lemma'][:20])

In [ ]:
# we might need to add in additional lemmatization that is specifc for our dataset: BR -> bedroom, or something like this
vocab_raw = set(all_tokens_nostop)
vocab_lemma = set(word for tokens in df['tokens_lemma'] for word in tokens)

print("Vocab size before lemmatization:", len(vocab_raw))
print("Vocab size after lemmatization :", len(vocab_lemma))

In [ ]:
# examine word frequency AFTER lemmzation
# word frequency 
common_words = Counter(vocab_lemma)
most_common = Counter(
    word for tokens in df['tokens_lemma'] for word in tokens
).most_common(20)

words, counts = zip(*most_common)

plt.barh(words, counts)
plt.gca().invert_yaxis()
plt.title("Top 20 Most Common Words (Lemmatized)")
plt.show()
# note 'BR' as the top word and 'bedroom' as the 4th --> these are representing the same words 

In [ ]:
# explore bigrams -- what words appear together most frequently 
bigram_list = [
    bigram
    for tokens in df['tokens_lemma']
    for bigram in bigrams(tokens)
]

Counter(bigram_list).most_common(50)

# note some additional cleaning needed for our dataset -- we don't care about 'Bedroom', 'king bed', 'queen bed', 'full kitchen', etc. 

In [ ]:
# examine the diversity of the words within our set 
df['unique_ratio'] = df['tokens_lemma'].apply(
    lambda x: len(set(x)) / len(x) if len(x) > 0 else 0
)

plt.hist(df['unique_ratio'], bins=30)
plt.title("Lexical Diversity of Airbnb Descriptions")
plt.show()

# Note: further exploring of this needed -- esp once we add in more dataset specific word removal

### Exploring: Keywords extraction

In [ ]:
# Test TFIDF approach 
# turn text into numbers -> keep only the useful words 
df['final_text'] = df['tokens_lemma'].apply(lambda x: ' '.join(x))

vectorizer = TfidfVectorizer(
    max_df=0.85,
    min_df=5,
    ngram_range=(1,2)   # unigrams + bigrams
)

tfidf_matrix = vectorizer.fit_transform(df['final_text'])
feature_names = vectorizer.get_feature_names_out()

In [ ]:
def extract_top_keywords(tfidf_row, feature_names, top_n=10):
    sorted_indices = np.argsort(tfidf_row.toarray()[0])[::-1]
    top_features = [feature_names[i] for i in sorted_indices[:top_n]]
    return top_features

df['keywords'] = [
    extract_top_keywords(tfidf_matrix[i], feature_names, top_n=10)
    for i in range(tfidf_matrix.shape[0])
]

In [ ]:
df['keywords']